In [1]:
import sympy as sp
import numpy as np
import sys

def phuong_phap_day_cung_sai_so_muc_tieu():
    print("=== CHƯƠNG TRÌNH TÌM NGHIỆM BẰNG PHƯƠNG PHÁP DÂY CUNG ===")
    print("(Công thức sai số mục tiêu)\n")
    
    try:
        # --- 0. Nhập liệu ---
        raw_f_str = input("Nhập hàm f(x) (ví dụ: x**3 - x - 1): ")
        f_str = raw_f_str.replace('^', '**') 
        
        a = float(input("Nhập đầu mút a: "))
        b = float(input("Nhập đầu mút b: "))
        
        if a > b:
            a, b = b, a
            
        epsilon = float(input("Nhập sai số mục tiêu epsilon (ví dụ: 1e-4): "))
        precision = int(input("Nhập số chữ số làm tròn xấp xỉ (precision): "))
        
        # --- 1. Xử lý toán học ---
        x = sp.Symbol('x')
        f_expr = sp.sympify(f_str)
        df_expr = sp.diff(f_expr, x)
        d2f_expr = sp.diff(df_expr, x)
        
        f = sp.lambdify(x, f_expr, 'numpy')
        df = sp.lambdify(x, df_expr, 'numpy')
        d2f = sp.lambdify(x, d2f_expr, 'numpy')
        
        xs = np.linspace(a, b, 1000)
        df_vals = df(xs)
        d2f_vals = d2f(xs)

        # --- 2. Kiểm tra điều kiện an toàn ---
        fa, fb = f(a), f(b)
        if np.isclose(fa, 0):
            print(f"\n=> KẾT QUẢ NGAY LẬP TỨC: Nghiệm chính là tại mút a = {a}")
            return
        if np.isclose(fb, 0):
            print(f"\n=> KẾT QUẢ NGAY LẬP TỨC: Nghiệm chính là tại mút b = {b}")
            return
            
        if fa * fb > 0:
            print(f"\n[!] LỖI: f({a}) và f({b}) cùng dấu. Không đảm bảo khoảng phân li nghiệm.")
            return

        if np.any(df_vals > 0) and np.any(df_vals < 0):
            print("\n[!] LỖI: Đạo hàm f'(x) đổi dấu trên [a, b]. Vui lòng thu hẹp khoảng.")
            return
            
        if np.any(d2f_vals > 0) and np.any(d2f_vals < 0):
            print("\n[!] LỖI: Đạo hàm f''(x) đổi dấu trên [a, b]. Vui lòng thu hẹp khoảng.")
            return

        # Tính toán m1
        abs_df_vals = np.abs(df_vals)
        m1 = np.min(abs_df_vals)
        
        if np.isclose(m1, 0, atol=1e-10):
            print("\n[!] LỖI CHÍ MẠNG: m1 = 0, gây lỗi chia cho 0. Vui lòng thu hẹp khoảng nghiệm.")
            return

        # Xác định điểm cố định d (điểm Fourier) và điểm xuất phát x0
        d2fa, d2fb = d2f(a), d2f(b)
        if fa * d2fa > 0:
            d, fd = a, fa
            x0, fx0 = b, fb
        else:
            d, fd = b, fb
            x0, fx0 = a, fa

        # --- 3. IN KẾT QUẢ MARKDOWN ---
        print("\n" + "="*50)
        print("### 1. Kiểm tra điều kiện và Các tham số ban đầu")
        print(f"* **Hàm số:** $f(x) = {sp.latex(f_expr)}$")
        print(f"* **Đạo hàm:** $f'(x) = {sp.latex(df_expr)}$ và $f''(x) = {sp.latex(d2f_expr)}$ xác định dấu không đổi.")
        print(f"* **Khoảng phân li nghiệm:** $f({a}) = {fa:.{precision}f}, f({b}) = {fb:.{precision}f} \\Rightarrow f({a})f({b}) < 0$")
        
        print("\n### 2. Xây dựng công thức và Đánh giá sai số")
        print(f"* **Hằng số đạo hàm:** $m_1 = \\min |f'(x)| = {m1:.{precision}f}$")
        print(f"* **Chọn điểm cố định $d$:** Vì $f({d}) \\cdot f''({d}) > 0$, chọn điểm Fourier $d = {d}$.")
        print(f"* **Chọn điểm xuất phát $x_0$:** Vì $f({x0}) \\cdot f({d}) < 0$, ta chọn $x_0 = {x0}$.")
        
        print("* **Công thức lặp:**")
        print(f"  $$x_n = x_{{n-1}} - \\frac{{f(x_{{n-1}})(x_{{n-1}} - {d})}}{{f(x_{{n-1}}) - f({d})}}$$")
        print("* **Công thức sai số mục tiêu:**")
        print(f"  $$\\Delta_n = \\frac{{|f(x_n)|}}{{m_1}} < {epsilon:.1e}$$")

        # --- 4. Tính toán xấp xỉ ---
        history = []
        xk, fxk = x0, fx0
        n = 1
        
        while True:
            if np.isclose(fxk, fd, atol=1e-15):
                print(f"\n[!] LỖI: Tại bước lặp {n}, f(x_n) tiến quá sát f(d). Thuật toán vỡ.")
                break
                
            x_next = xk - (fxk * (xk - d)) / (fxk - fd)
            fx_next = f(x_next)
            
            # Đánh giá sai số mục tiêu
            error = abs(fx_next) / m1
            
            history.append({
                'n': n, 
                'x_prev': xk,
                'fx_prev': fxk,
                'x_next': x_next, 
                'fx_next': fx_next,
                'error': error
            })
            
            if error <= epsilon or n > 100:
                break
                
            xk, fxk = x_next, fx_next
            n += 1
            
        total_steps = len(history)

        print("\n### 3. Chi tiết các bước lặp")
        
        # 2 BƯỚC ĐẦU TIÊN
        print("**Hai bước lặp đầu tiên:**")
        for i in range(min(2, total_steps)):
            step = history[i]
            print(f"* **$n={step['n']}$:**")
            print(f"  $$x_{step['n']} = {step['x_prev']:.{precision}f} - \\frac{{{step['fx_prev']:.{precision}f} \\times ({step['x_prev']:.{precision}f} - {d})}}{{{step['fx_prev']:.{precision}f} - {fd:.{precision}f}}} = {step['x_next']:.{precision}f}$$")
            print(f"  $$\\Delta_{step['n']} = \\frac{{|{step['fx_next']:.{precision}f}|}}{{{m1:.{precision}f}}} = {step['error']:.{precision}g}$$")

        # 2 BƯỚC CUỐI CÙNG
        if total_steps > 2:
            print("\n**Hai bước lặp cuối cùng:**")
            for i in range(max(2, total_steps - 2), total_steps):
                step = history[i]
                print(f"* **$n={step['n']}$:**")
                print(f"  $$x_{step['n']} = {step['x_prev']:.{precision}f} - \\frac{{{step['fx_prev']:.{precision}f} \\times ({step['x_prev']:.{precision}f} - {d})}}{{{step['fx_prev']:.{precision}f} - {fd:.{precision}f}}} = {step['x_next']:.{precision}f}$$")
                print(f"  $$\\Delta_{step['n']} = \\frac{{|{step['fx_next']:.{precision}f}|}}{{{m1:.{precision}f}}} = {step['error']:.{precision}g}$$")

        print("\n### 4. Bảng lặp rút gọn\n")
        print(f"| $n$ | $x_{{n-1}}$ | $f(x_{{n-1}})$ | $x_n$ | $f(x_n)$ | $\\Delta_n$ | ")
        print("| :--- | :--- | :--- | :--- | :--- | :--- | ")
        
        for i in range(min(3, total_steps)):
            s = history[i]
            print(f"| {s['n']} | {s['x_prev']:.{precision}f} | {s['fx_prev']:.{precision}f} | {s['x_next']:.{precision}f} | {s['fx_next']:.{precision}f} | {s['error']:.{precision}g} |")
            
        if total_steps > 6:
            print("| ... | ... | ... | ... | ... | ... |")
            
        if total_steps > 3:
            for i in range(max(3, total_steps - 3), total_steps):
                s = history[i]
                print(f"| {s['n']} | {s['x_prev']:.{precision}f} | {s['fx_prev']:.{precision}f} | {s['x_next']:.{precision}f} | {s['fx_next']:.{precision}f} | {s['error']:.{precision}g} |")

        print(f"\n=> **KẾT LUẬN:** $x \\approx {history[-1]['x_next']:.{precision}f}$")

    except Exception as e:
        print(f"\n[!] CÓ LỖI XẢY RA: {e}")

if __name__ == "__main__":
    phuong_phap_day_cung_sai_so_muc_tieu()
    input("\nNhấn Enter để thoát chương trình...")

=== CHƯƠNG TRÌNH TÌM NGHIỆM BẰNG PHƯƠNG PHÁP DÂY CUNG ===
(Công thức sai số mục tiêu)


### 1. Kiểm tra điều kiện và Các tham số ban đầu
* **Hàm số:** $f(x) = x^{3} + 3 x^{2} - 1$
* **Đạo hàm:** $f'(x) = 3 x^{2} + 6 x$ và $f''(x) = 6 x + 6$ xác định dấu không đổi.
* **Khoảng phân li nghiệm:** $f(0.1) = -0.9690000, f(1.0) = 3.0000000 \Rightarrow f(0.1)f(1.0) < 0$

### 2. Xây dựng công thức và Đánh giá sai số
* **Hằng số đạo hàm:** $m_1 = \min |f'(x)| = 0.6300000$
* **Chọn điểm cố định $d$:** Vì $f(1.0) \cdot f''(1.0) > 0$, chọn điểm Fourier $d = 1.0$.
* **Chọn điểm xuất phát $x_0$:** Vì $f(0.1) \cdot f(1.0) < 0$, ta chọn $x_0 = 0.1$.
* **Công thức lặp:**
  $$x_n = x_{n-1} - \frac{f(x_{n-1})(x_{n-1} - 1.0)}{f(x_{n-1}) - f(1.0)}$$
* **Công thức sai số mục tiêu:**
  $$\Delta_n = \frac{|f(x_n)|}{m_1} < 1.0e-07$$

### 3. Chi tiết các bước lặp
**Hai bước lặp đầu tiên:**
* **$n=1$:**
  $$x_1 = 0.1000000 - \frac{-0.9690000 \times (0.1000000 - 1.0)}{-0.9690000 - 3.0000000} = 0.3197279$$
  $$\Del